# Fine-tuning to follow instructions
The last chapter covers the instruction fine-tuning process for LLMs. This is one of the main techniques behind developing chatbot applications or personal assistants.

### 7.2 Preparing a dataset for supervised instruction fine-tuning
Step one is to prepare the dataset so that the model can learn the way we want it to. 

In [2]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    """Function that downloads and loads dataset, skips download if already donwloaded"""
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text_data)

    with open(file_path, "r") as f:
        data = json.load(f)
    return data


In [6]:
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print("number of entries: ", len(data))

print(f"Example entry:\n{data[50]}")
print(f"Another example entry:\n{data[999]}")

number of entries:  1100
Example entry:
{'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}
Another example entry:
{'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


In [7]:
def format_input(entry):
    """Function that converts the entries in the dataset to Alpaca-style input format"""
    instruction_text = (
        f"Bellow is an instruction that describes a task. "
        f"Write a response that appropriately completes the request. "
        f"\n\n### Instruction:\n{entry['instruction']}"
        )

    input_text = (
        f"\n\n### Input:\n{entry["input"]}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [8]:
# Test it on entry 50
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input+desired_response)

Bellow is an instruction that describes a task. Write a response that appropriately completes the request. 

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [9]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input+desired_response)

Bellow is an instruction that describes a task. Write a response that appropriately completes the request. 

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [12]:
# Divide dataset into training, validation and testing
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]   # 85%
test_data = data[train_portion:train_portion + test_portion]    # 10%
val_data = data[train_portion + test_portion:]   # 5%

print("Training set length: ", len(train_data))
print("Test set length: ", len(test_data))
print("Validation set length: ", len(val_data))

Training set length:  935
Test set length:  110
Validation set length:  55


### 7.3 Organizing data into training batches
Having succesfully divided the data into training, validation and testing data it's now time to organize it so that we can use batch training

In [ ]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry) # Step 1: Format dataset entry
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append( 
                tokenizer.encode(full_text) # Step 2: Tokenize formatted entry
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [15]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [21]:
# Step 3: Pad shorter messages to achieve uniform padding for each batch

def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    bacth_max_length = max(len(item)+1 for item in batch) # Find longest message in batch
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (new_item + [pad_token_id] * (bacth_max_length - len(new_item)))

        inputs = torch.tensor(padded[:-1]) # Remove extra padded token added earlier
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor


In [23]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [24]:
# Step 4: Update collate function to add list of  target token IDs for training
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    bacth_max_length = max(len(item)+1 for item in batch) # Find longest message in batch
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded  = (new_item + [pad_token_id] * (bacth_max_length - len(new_item)))

        inputs  = torch.tensor(padded[:-1]) # Remove extra padded token added earlier
        targets = torch.tensor(padded[1:])  # Shifts +1 to the right for targets
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device) 
    return inputs_tensor, targets_tensor

In [28]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs, "\n")
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]]) 

tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [29]:
# Step 5: Replace padding tokens with placeholders in targets list
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    bacth_max_length = max(len(item)+1 for item in batch) # Find longest message in batch
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded  = (new_item + [pad_token_id] * (bacth_max_length - len(new_item)))

        inputs  = torch.tensor(padded[:-1]) # Remove extra padded token added earlier
        targets = torch.tensor(padded[1:])  # Shifts +1 to the right for targets

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1: # Replace all bbut first padding token in targets
            targets[indices[1:]] = ignore_index 

        if allowed_max_length is not None: # Truncate to maximum sequence length
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device) 
    return inputs_tensor, targets_tensor

In [31]:
inputs, targets = custom_collate_fn(batch)
print(inputs, "\n")
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]]) 

tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [ ]:
# Why do we need the mask? 
logits_1 = torch.tensor(
    [[-1.0, 1.0],    # Predictions first token
     [-0.5, 1.5]]    # Predictions second token 
)
targets_1 = torch.tensor([0, 1]) # Correct token indices to generate
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


In [ ]:
# Adding another token ID, loss goes down
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]     
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


In [ ]:
# Now lets replace the third token ID with -100
# -100 is automatically set to be ignored by the cross_entropy function
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


In [ ]:
"""The reason for masking out padding tokens except for the first one is that we don't want 
the model to learn from these artifical tokens, hence we ignore them in the loss calculation.
But since we want the model to learn when a real sequence should end, we keep the first padding token.

It is also common (but not required) to mask out the target token IDs that correspond to the instruction
so that the loss is computed only on the generated response target."""

### 7.4 Creating data loaders for instruction dataset
Once again we create dataloaders for fine-tuning the model

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [39]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [40]:
from torch.utils.data import DataLoader

num_workers = 2 # 0
batch_size  = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader  = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader  = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader  = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:


### 7.5 Loading a pretrained LLM
Load the pretrained GPT model we want to finetune